# Fine-tuning LoRA — Qwen2.5-7B-Instruct FR ↔ Zarma

**Avant de lancer** : `Runtime > Change runtime type > T4 GPU` (ou mieux si Colab Pro : A100/L4).

Ce notebook clone le dépôt, installe les dépendances, régénère le dataset de fine-tuning à partir des CSV consolidés, puis lance `scripts/train_lora.py` (QLoRA 4-bit).

In [ ]:
!nvidia-smi

In [ ]:
import os

REPO_DIR = "/content/zarma_ia_project"
if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
    !git pull
else:
    !git clone https://github.com/BiaoMoussa/zarma_ia_project.git {REPO_DIR}
    %cd {REPO_DIR}

In [ ]:
# peft/accelerate/bitsandbytes peuvent tirer torch comme dépendance transitive et
# le mettre à jour, même sans le lister nous-mêmes — ce qui casse à nouveau la
# compatibilité avec le torchvision/CUDA déjà installés par Colab. On fige donc
# torch/torchvision/torchaudio à leurs versions actuelles via un fichier de
# contraintes pip, qui interdit tout upgrade transitif.
!python -c "
import torch, torchvision
versions = [f'torch=={torch.__version__}', f'torchvision=={torchvision.__version__}']
try:
    import torchaudio
    versions.append(f'torchaudio=={torchaudio.__version__}')
except ImportError:
    pass
open('/tmp/torch-constraints.txt', 'w').write('\n'.join(versions) + '\n')
print(open('/tmp/torch-constraints.txt').read())
"
!grep -v '^torch==' requirements.txt > /tmp/requirements-colab.txt
!pip install -q -r /tmp/requirements-colab.txt -r requirements-train.txt -c /tmp/torch-constraints.txt

# Vérification : torch/torchvision doivent rester importables et cohérents
!python -c "import torch, torchvision; print('torch', torch.__version__, '| torchvision', torchvision.__version__, '| CUDA disponible:', torch.cuda.is_available())"

In [ ]:
# Régénère zarma_corpus/finetune/{train,validation,test}.jsonl à partir des CSV
# consolidés déjà présents dans le dépôt (pas besoin de les committer, c'est
# déterministe — seed fixe dans le script).
!python scripts/prepare_finetune_dataset.py

## (Recommandé) Monter Google Drive

Les sessions Colab sont éphémères : si la session se déconnecte, tout ce qui n'est
pas sur Drive est perdu. On sauvegardera l'adaptateur LoRA sur Drive à la fin.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!python scripts/train_lora.py

In [ ]:
# Sauvegarde l'adaptateur entraîné sur Drive (persiste au-delà de la session Colab)
!mkdir -p /content/drive/MyDrive/zarma_lora_adapters
!cp -r lora_adapters/qwen2.5-7b-zarma-fr /content/drive/MyDrive/zarma_lora_adapters/
print("Sauvegardé sur Drive : MyDrive/zarma_lora_adapters/qwen2.5-7b-zarma-fr")

## Test rapide d'inférence

Charge le modèle de base + l'adaptateur LoRA fraîchement entraîné et teste une traduction.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
ADAPTER_DIR = "lora_adapters/qwen2.5-7b-zarma-fr"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map="auto"
)
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)

messages = [
    {"role": "system", "content": "Tu es un traducteur expert français ↔ zarma (Djerma), une langue parlée au Niger. Tu traduis fidèlement, en respectant le sens et le registre du texte source."},
    {"role": "user", "content": "Traduis en zarma : Les enfants sont nus."},
]
inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(model.device)
outputs = model.generate(inputs, max_new_tokens=100)
print(tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True))